In [1]:
import base64
import requests
import os
import glob
import json
import time
import pandas as pd
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
# Load environment variables
load_dotenv()

True

In [3]:
# Initialize OpenAI client with API key
client = OpenAI()
OpenAI.api_key = os.getenv('OPENAI_API_KEY')

In [4]:
# Load prompts from the JSON file
with open('../config/prompts.json', 'r') as file:
    prompts = json.load(file)

In [5]:
class Advertisement(BaseModel):
    Ad_Name: str
    Product: str
    Company: str
    Ad_Description: str
    Ad_Placement: str

class AdvertisementList(BaseModel):
    Advertisements: list['Advertisement']

class NewsArticleData(BaseModel):
    Headline: str
    Article_Text: str
    Published_Date: str
    Publishing_Agency_Name: str
    Author_Names: list[str]
    Keywords_Tags: list[str]
    Related_Article_Links: list[str]
    Sentiment: str
    Tone: str
    Narrative: str        

In [6]:
def encode_images_to_base64(folder_path):
    """
    Load all images from a specified folder and encode them in Base64.

    Args:
        folder_path (str): The path to the folder containing images.

    Returns:
        list: A list of Base64 encoded images.
    """
    images = []
    for filename in glob.glob(os.path.join(folder_path, '*.*')):
        with open(filename, 'rb') as image_file:
            b64_image = base64.b64encode(image_file.read()).decode('utf-8')
            images.append(b64_image)
    return images

In [7]:
def generate_ad_completions(prompt, b64_images, model='gpt-4o', max_tokens=300):
    """
    Generates advertisement completions based on the provided prompt and base64-encoded images.

    Args:
    prompt (str): The text prompt to generate advertisements.
    b64_images (list): A list of base64-encoded images.
    model (str): The model to use for generating completions. Default is 'gpt-4o'.
    max_tokens (int): The maximum number of tokens for the response. Default is 300.

    Returns:
    list: A list of advertisement completions.
    """
    ad_completions = []  # Initialize an empty list to store the advertisement completions
    
    # Loop through each base64-encoded image
    for b64_image in b64_images:
        # Generate a chat completion using the provided model
        completion = client.beta.chat.completions.parse(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64_image}"
                            }
                        }
                    ]
                }
            ],
            response_format=AdvertisementList,
            max_tokens=max_tokens
        )
        
        # Parse the completion result and append to the list
        ad_content = completion.choices[0].message.parsed
        ad_completions.append(ad_content)
    
    return ad_completions  # Return the list of advertisement completions


In [8]:
def generate_news_article_completions(prompts, b64_images, model='gpt-4o', max_tokens=2000):
    """
    Sends a request to the OpenAI API to generate news article completions based on the given prompts
    and base64-encoded images.

    Args:
        prompts (list): A list of text prompts to be used by the model.
        b64_images (list): A list of base64-encoded images.
        model (str): The model name to be used. Defaults to 'gpt-4o'.
        max_tokens (int): The maximum number of tokens to generate. Defaults to 2000.

    Returns:
        dict: A parsed response containing the generated news article.
    """ 
    # Create the payload for the API request
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt} for prompt in prompts
                ] + [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{b64_image}"
                        }
                    } for b64_image in b64_images
                ]
            }
        ],
        response_format=NewsArticleData,
        max_tokens=max_tokens
    )

    # Return the parsed message containing the generated news article
    return completion.choices[0].message.parsed

In [9]:
def generate_dataframe_from_folders(root_path, prompts):
    """
    Generate DataFrames containing article and advertisement data by processing folders with images and prompts.

    Args:
        root_path (str): The root directory containing folders with images.
        prompts (dict): A dictionary containing text prompts for news article and advertisement extraction.

    Returns:
        tuple: Two DataFrames, one containing the processed article data and the other containing advertisement data.
    """
    articles_data = []
    ads_data = []
    
    # Iterate through each folder in the root directory
    for folder in os.listdir(root_path):
        folder_path = os.path.join(root_path, folder)
        
        # Load all images from the folder and encode them to base64
        b64_images = encode_images_to_base64(folder_path)
        
        # Generate news article data using the corresponding prompts
        news_article_data = generate_news_article_completions(
            prompts["news_article_data_extraction_prompts"], b64_images, model='gpt-4o-mini', max_tokens=2000
        )
        
        # Generate advertisement data using the corresponding prompt
        advertisement_data = generate_ad_completions(
            prompts["advertisement_extraction_prompt"], b64_images, model='gpt-4o-mini', max_tokens=2000
        )
        
        # Process and store the advertisement data
        for ads in advertisement_data:
            for ad in ads.Advertisements:
                ad_dict = {
                    'Folder_Number': folder,
                    'Ad_Name': ad.Ad_Name,
                    'Product': ad.Product,
                    'Company': ad.Company,
                    'Ad_Description': ad.Ad_Description,
                    'Ad_Placement': ad.Ad_Placement
                }
                ads_data.append(ad_dict)
        
        # Construct and store the article data for the current folder
        article_dict = {
            'Folder_Number': folder,
            'Headline': news_article_data.Headline,
            'Article_Text': news_article_data.Article_Text,
            'Published_Date': news_article_data.Published_Date,
            'Publishing_Agency_Name': news_article_data.Publishing_Agency_Name,
            'Author_Names': news_article_data.Author_Names,
            'Keywords_Tags': news_article_data.Keywords_Tags,
            'Related_Article_Links': news_article_data.Related_Article_Links,
            'Sentiment': news_article_data.Sentiment,
            'Tone': news_article_data.Tone,
            'Narrative': news_article_data.Narrative
        }
        articles_data.append(article_dict)

    # Return DataFrames containing the processed article and advertisement data
    return pd.DataFrame(articles_data), pd.DataFrame(ads_data)


In [10]:
root_path = '../screenshot_data/24-05-2024'

# Assuming prompts is a dictionary containing the appropriate prompts
article_data_df, ad_data_df = generate_dataframe_from_folders(root_path, prompts)

In [11]:
article_data_df

,Folder_Number,Headline,Article_Text,Published_Date,Publishing_Agency_Name,Author_Names,Keywords_Tags,Related_Article_Links,Sentiment,Tone,Narrative
0,130,Thousands of illegal Muslims occupied the Gree...,The ND said they would allow them to build mos...,09-03-2024,PRONEWS,[Not specifically mentioned],"[illegal, Muslims, squares, prayer, Greece]","[See the photo that was referenced in ""X""]",Negative,Concerned,The article conveys a critical perspective reg...
1,139,When Ursula's grandmother greeted A. Hitler as...,A viral social media post features a photo dep...,19-03-2024,PRONEWS,[],"[Ursula von der Leyen, Adolf Hitler, Nazi party]",[],Negative,Historical,The article reflects on the historical context...
2,147,"In Ukraine, Russia is ""raining"" – What Aleksei...",The article discusses the ongoing responsibili...,23-03-2024,Mega TV,[Unknown],"[Ukraine, Russia, Aleksei Danilov, terrorist a...","[Related Article 1, Related Article 2]",Negative,Alarmist,The article conveys a critical stance towards ...
3,155,It’s ‘time to stop using terms like long COVID...,Follow the science — right out the door.\n\nTh...,14-03-2024,New York Post,[David Landsel],"[long COVID, health agency, unnecessary fear, ...","[5 dead, dozens hospitalized as ‘parrot fever’...",Negative,Critical,"The article critiques the term 'long COVID,' s..."
4,157,Is it long COVID or long vax? Does the governm...,America's health agencies need to snap into ac...,01-10-2023,The Hill,[],[],[California legalizes human composting by 2027...,Neutral,Informative,The article emphasizes the need for U.S. healt...
5,214,French soldiers killed in Russian bombing in O...,"In particular, ""in the area of the Romania-Ukr...",16-03-2024,PRONEWS,[],"[military, Ukraine, Russia, NATO]",[],Negative,Alarmist,The article conveys a message of concern about...
6,25,[ŽT] Syrian Son Welcomes Russians with Victory...,Alexander Syrian's son Ivan Syrian arrived in ...,21-02-2024,MTPC,[Aleksandras Syriskis],"[Syria, Russia, Victory, Australia, Consulate]","[Kara Ukrainoje, nuolat plintančios žinios, Ko...",Neutral,Informative,The article discusses Ivan Syrian's position r...
7,8,Sánchez gave Morocco 400 police cars and 90 ca...,"In this delivery, the Government gifted the Mo...",24-05-2024,okdiario,[Not provided],"[Sánchez, Morocco, Civil Guard, Police Cars, G...",[N/A],Negative,Critical,The article criticizes the Spanish government'...
8,88,"Samiti për Ukrainën në Tiranë, Zelensky jep me...","The summit held in Tirana focused on Ukraine, ...",28-02-2024,Pamfleti,[Shkruar nga Pamfleti],"[Ukraine, Tirana Summit, Zelensky, Albania]","[New Developments in Ukraine, International Re...",Positive,Encouraging,"The article conveys a pro-Ukrainian message, h..."
9,99,US's secret 'Turkey map' opened the way for an...,A secret map revealing Turkey's interests in b...,04-03-2024,Milliyet,"[Özay Şendir, Güneri Civaağlı, Didem Özel Tümer]","[Yunan Adaları, Ege, Türkiye]","[Turkey’s response to the US Congress report, ...",Negative,Fear-mongering,The article conveys a strong anti-American sen...


In [13]:
ad_data_df.head(20)

,Folder_Number,Ad_Name,Product,Company,Ad_Description,Ad_Placement
0,130,Drive Magazine,Drive magazine subscription or issue,Drive,Special subscription offers and issues related...,top section before the headline
1,130,History Magazine,History magazine subscription or issue,History,Special subscription offers and issues on vari...,top section before the headline
2,130,Calculator,Body Fat Calculator,Not specified,Calculates the time needed to fast to achieve ...,Left sidebar
3,130,Braun Minipimer,Minipimer (Hand Blender),Braun,Participate in the sweepstakes to win a Braun ...,Middle section
4,130,Advertisement,Not specified,Not specified,with only a very small percentage of aneuploid...,"center of the page, as a banner advertisement"
5,130,Simyo,Mobile Service,Simyo,5G mobile plans with unlimited calls and a pro...,sidebar
6,130,Nuevo PEUGEOT 2008 HYBRID,Car,Peugeot,Promotion of the new Peugeot 2008 hybrid vehicle.,banner
7,130,Nuevo PEUGEOT 2008 HYBRID,Car,Peugeot,"Introducing the new Peugeot 2008 Hybrid, a mod...",Right sidebar
8,130,¡Hola 5! 30GB y llamadas ilimitadas por 10€,Mobile Plan,simyo,Get 30GB and unlimited calls for just 10 euros...,Right sidebar
9,130,Simyo,5G Mobile Plans,Simyo,Now 5G on all our plans for the same price.,"Banner ad, middle section of the page."


In [14]:
article_data_df.to_csv("../datasets/articles_dataset_24_05_2024.csv",index=False)
ad_data_df.to_csv("../datasets/ads_dataset_24_05_2024.csv",index=False)